## Establishing Player 'Groups'

FBref labels every player with a tag like MF or DF,MF, but these mix together players with very different jobs. A defensive midfielder and an attacking midfielder can both get tagged "MF," even though one's job is to break up play and the other's is to create chances. Rating them the same way would hide that difference.

So instead of using FBref's raw tags directly, I split players into six roles based on which tags they have: Forward, Attacking Midfielder, Midfielder, Defensive Midfielder, Defender, and Goalkeeper. For example, MF,FW (leans attacking) is treated separately from MF,DF (leans defensive).

Worth noting: this is based on where a player lines up, not necessarily their true role — a manager might use someone differently than their label suggests. But it's a solid, data-driven starting point.

In [163]:
def get_position_group(pos):
    if pd.isna(pos):
        return 'Unknown'
    if pos in ['FW', 'FW,MF']:
        return 'Forward'
    elif pos == 'MF,FW':
        return 'Attacking Midfielder'
    elif pos == 'MF':
        return 'Midfielder'
    elif pos == 'MF,DF':
        return 'Defensive Midfielder'
    elif pos in ['DF', 'DF,MF', 'DF,FW']:
        return 'Defender'
    elif pos == 'GK':
        return 'Goalkeeper'
    return 'Unknown'

df['position_group'] = df['Pos'].apply(get_position_group)
print(df['position_group'].value_counts())

position_group
Midfielder              1065
Defender                 925
Forward                  558
Attacking Midfielder     221
Goalkeeper               212
Defensive Midfielder     133
Unknown                    1
Name: count, dtype: int64


## Forward analysis

The first thing to do here is to select only players who have reached above a certain threshold of minutes. This eliminates any anomalies that perhaps played one game and scored once, averaging one goal per game. 

In [164]:
MIN_MINUTES = 900
forwards = df[(df['position_group'] == 'Forward') & (df['Min'] >= MIN_MINUTES)].copy()
print(len(forwards))

195


Forwards need to make dangerous and perfectly timed runs to beat the defense. It would be unfair to penalise every player who makes a bad run.

However, I have decided to only penlalise those in the 85th percentile for OFF_p90. This should penliase those who are often making worse decisions and causing stops in play.

In [165]:
forwards['Off_p90'] = forwards['Off'] / forwards['90s']
offside_threshold_p90 = forwards['Off_p90'].quantile(0.85)
forwards['offside_penalty'] = (forwards['Off_p90'] - offside_threshold_p90).clip(lower=0)

In [166]:
forwards['Fld_p90'] = forwards['Fld'] / forwards['90s']
forwards['Crs_p90'] = forwards['Crs'] / forwards['90s']
forwards['CrdR_p90'] = forwards['CrdR'] / forwards['90s']
forwards['BigChancesCreated_p90'] = forwards['BigChancesCreated'] / forwards['90s']

print(forwards[['Fld_p90', 'Crs_p90', 'CrdR_p90', 'BigChancesCreated_p90']].describe())

          Fld_p90     Crs_p90    CrdR_p90  BigChancesCreated_p90
count  195.000000  195.000000  195.000000             167.000000
mean     1.474457    0.976094    0.006611               0.201374
std      0.719053    1.297010    0.020065               0.140671
min      0.109290    0.000000    0.000000               0.031746
25%      0.935921    0.277802    0.000000               0.091955
50%      1.370968    0.528967    0.000000               0.174419
75%      1.919302    1.018519    0.000000               0.258902
max      4.117647    7.553957    0.111111               0.935252


Next, its important to fill zeros for players who didnt reach the '1' threshold for certain stats. Fotmob doesnt show players with less than 1 of the statistic, therefore anyone with the stat NaN in these categories have 0.

In [167]:
zero_fill_cols = ['BigChancesCreated_p90', 'creative_output_p90', 'Crs_p90']

for col in zero_fill_cols:
    forwards[col] = forwards[col].fillna(0)

print(forwards[zero_fill_cols].describe())

       BigChancesCreated_p90  creative_output_p90     Crs_p90
count             195.000000           195.000000  195.000000
mean                0.172459             0.789231    0.976094
std                 0.148137             0.499006    1.297010
min                 0.000000             0.000000    0.000000
25%                 0.071852             0.500000    0.277802
50%                 0.152672             0.700000    0.528967
75%                 0.243683             1.000000    1.018519
max                 0.935252             3.000000    7.553957


### What "good" means for a forward

Rather than multiple different weights, the tool uses TOPSIS - the same used in a real published study evaluating Premier League strikers (Kolbowicz et al., 2024)

The idea is to make both a 'perfect-case', and 'worst-case' forward based on the highest and lowest value each stat actually reaches across the dataset, so both reference points are grounded in real performances, not guesses.

In [168]:
central_forward_criteria = {
    'Gls_p90': 'profit',
    'G/Sh': 'profit',
    'SoT%': 'profit',
    'Sh/90': 'profit',
    'Ast_p90': 'profit',
    'pct_of_team_goals': 'profit',
    'Fld_p90': 'profit',
    'league_success_score': 'profit',
    'CrdR_p90': 'cost',
    'offside_penalty': 'cost',
}

wide_creator_criteria = {
    'creative_output_p90': 'profit',
    'BigChancesCreated_p90': 'profit',
    'Ast_p90': 'profit',
    'Crs_p90': 'profit',
    'Gls_p90': 'profit',
    'pct_of_team_assists': 'profit',
    'Fld_p90': 'profit',
    'league_success_score': 'profit',
    'CrdR_p90': 'cost',
    'offside_penalty': 'cost',
}

print("Central Forward criteria:", list(central_forward_criteria.keys()))
print("Wide Creator criteria:", list(wide_creator_criteria.keys()))

Central Forward criteria: ['Gls_p90', 'G/Sh', 'SoT%', 'Sh/90', 'Ast_p90', 'pct_of_team_goals', 'Fld_p90', 'league_success_score', 'CrdR_p90', 'offside_penalty']
Wide Creator criteria: ['creative_output_p90', 'BigChancesCreated_p90', 'Ast_p90', 'Crs_p90', 'Gls_p90', 'pct_of_team_assists', 'Fld_p90', 'league_success_score', 'CrdR_p90', 'offside_penalty']


The next step is to split the FW players into 2 categories: Central forwards and Wide forwards. KMeans clustering allows a machine learning algroithm to group players who are similar in certain abilities. For example a forward with the aim to perform accurate crosses and beating their defender should not be penlised heavily for not scoring goals.

In [169]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cluster_features = ['Gls_p90', 'Ast_p90', 'Crs', 'creative_output_p90']

cluster_data = forwards[cluster_features].fillna(0)

scaler = StandardScaler()
cluster_data_scaled = scaler.fit_transform(cluster_data)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
forwards['cluster'] = kmeans.fit_predict(cluster_data_scaled)

print(forwards['cluster'].value_counts())

cluster
1    153
0     42
Name: count, dtype: int64


In [170]:
def normalize_column(series, criterion_type):
    if criterion_type == 'profit':
        return (series - series.min()) / (series.max() - series.min())
    else:  # cost
        return (series.max() - series) / (series.max() - series.min())

central_forwards = forwards[forwards['cluster'] == 1].copy()
wide_creators = forwards[forwards['cluster'] == 0].copy()

for col, criterion_type in central_forward_criteria.items():
    central_forwards[col + '_norm'] = normalize_column(central_forwards[col], criterion_type)

for col, criterion_type in wide_creator_criteria.items():
    wide_creators[col + '_norm'] = normalize_column(wide_creators[col], criterion_type)

print("Central Forward normalized stats:")
print(central_forwards[[c + '_norm' for c in central_forward_criteria.keys()]].describe())

print("\nWide Forward normalized stats:")
print(wide_creators[[c + '_norm' for c in wide_creator_criteria.keys()]].describe())

Central Forward normalized stats:
       Gls_p90_norm   G/Sh_norm   SoT%_norm  Sh/90_norm  Ast_p90_norm  \
count    153.000000  153.000000  153.000000  153.000000    153.000000   
mean       0.384464    0.375129    0.511397    0.439254      0.334060   
std        0.193873    0.169730    0.174860    0.183613      0.228922   
min        0.000000    0.000000    0.000000    0.000000      0.000000   
25%        0.250000    0.263158    0.398917    0.311346      0.185185   
50%        0.364583    0.368421    0.512635    0.414248      0.333333   
75%        0.489583    0.473684    0.617329    0.540897      0.481481   
max        1.000000    1.000000    1.000000    1.000000      1.000000   

       pct_of_team_goals_norm  Fld_p90_norm  league_success_score_norm  \
count              153.000000    153.000000                 153.000000   
mean                 0.325647      0.335238                   0.460131   
std                  0.206874      0.170929                   0.300304   
min         

## Forwards ratings

Now its time to build the ideal and worst-case player, and give a rating to the players. Every players distance to this criteria is measured and then given a final ranking score (1 being perfect, 0 being worst).

### Mathematical Formulation of the Rating System

**Step 1 — Decision Matrix**

Let $x_{ij}$ represent the raw value of criterion $i$ for player $j$, where $i = 1, \dots, n$ (the number of criteria) and $j = 1, \dots, m$ (the number of players).

**Step 2 — Normalization**

Each criterion is normalized to a $[0,1]$ range. For **profit** criteria (higher is better):

$$v_{ij} = \frac{x_{ij} - \min(x_i)}{\max(x_i) - \min(x_i)}$$

For **cost** criteria (lower is better, e.g. red cards):

$$v_{ij} = \frac{\max(x_i) - x_{ij}}{\max(x_i) - \min(x_i)}$$

**Step 3 — Weighting**

Since criteria are weighted equally, each weight is:

$$w_i = \frac{1}{n}$$

giving the weighted normalized value:

$$u_{ij} = w_i \cdot v_{ij}$$

**Step 4 — Ideal and Worst-Case Solutions**

$$u_i^{*} = \max_j(u_{ij}), \qquad u_i^{-} = \min_j(u_{ij})$$

**Step 5 — Euclidean Distance to Ideal and Worst-Case**

$$D_j^{*} = \sqrt{\sum_{i=1}^{n} \left(u_{ij} - u_i^{*}\right)^2}$$

$$D_j^{-} = \sqrt{\sum_{i=1}^{n} \left(u_{ij} - u_i^{-}\right)^2}$$

**Step 6 — Relative Closeness (Final Rating)**

$$C_j^{*} = \frac{D_j^{-}}{D_j^{*} + D_j^{-}}$$

The final rating is $C_j^{*} \times 100$, bounded between 0 and 100.

In [171]:
import numpy as np

def calculate_topsis(df, criteria):
    norm_cols = [col + '_norm' for col in criteria.keys()]
    weight = 1 / len(criteria)

    weighted = df[norm_cols] * weight

    ideal = weighted.max()
    worst = weighted.min()

    dist_to_ideal = np.sqrt(((weighted - ideal) ** 2).sum(axis=1))
    dist_to_worst = np.sqrt(((weighted - worst) ** 2).sum(axis=1))

    topsis_score = dist_to_worst / (dist_to_ideal + dist_to_worst)
    return topsis_score

central_forwards['rating'] = (calculate_topsis(central_forwards, central_forward_criteria) * 100).round(1)

print(central_forwards[['Player', 'Squad', 'league', 'Gls_p90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

          Player          Squad         league  Gls_p90  rating
          Mikael            CRB Brazil Serie B     0.79    67.6
 Tawfik Bentayeb         Troyes        Ligue 2     0.83    64.2
    Ivan Prtajin Kaiserslautern  2. Bundesliga     0.96    63.3
    Bruno Santos       Londrina Brazil Serie B     0.71    63.2
 Oliver McBurnie      Hull City   Championship     0.53    63.0
 Andrea Adorante        Venezia        Serie B     0.72    62.8
    Isac Lidberg   Darmstadt 98  2. Bundesliga     0.61    62.0
 Joel Pohjanpalo        Palermo        Serie B     0.66    61.9
    Ross Stewart    Southampton   Championship     0.73    61.6
Mateusz Żukowski      Magdeburg  2. Bundesliga     0.83    61.2
   Anselmo Ramon          Goiás Brazil Serie B     0.49    60.4
    Žan Vipotnik   Swansea City   Championship     0.70    59.3
     Ellis Simms  Coventry City   Championship     0.71    59.3
  Tommaso Biasci       Avellino        Serie B     0.47    59.1
     Noel Futkeu Greuther Fürth  2. Bund

In [172]:
wide_creators['rating'] = (calculate_topsis(wide_creators, wide_creator_criteria) * 100).round(1)

print("Top 15 Wide Forwards:")
print(wide_creators[['Player', 'Squad', 'league', 'creative_output_p90', 'Crs_p90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

Top 15 Wide Forwards:
             Player         Squad         league  creative_output_p90  Crs_p90  rating
         Barış Atik     Magdeburg  2. Bundesliga                  3.0 5.408805    62.2
      Fabio Maistro   Juve Stabia        Serie B                  2.0 7.553957    61.0
        John Yeboah       Venezia        Serie B                  2.1 1.211073    57.5
 Filippo Pittarello     Catanzaro        Serie B                  1.0 0.291262    57.0
      Irvin Cardona Saint-Étienne        Ligue 2                  2.2 3.122172    56.9
 Cristian Buonaiuto        Padova        Serie B                  2.0 5.047619    56.8
       Abdel Hbouch     Annecy FC        Ligue 2                  2.5 6.621622    55.9
              Pablo      Operário Brazil Serie B                  1.8 0.317460    55.6
        Amine Hemia      Guingamp        Ligue 2                  1.2 4.500000    55.3
Zuriko Davitashvili Saint-Étienne        Ligue 2                  1.7 2.638298    55.2
             Robson N

## Saving forwards ratings

In [173]:
central_forwards['forward_subtype'] = 'Central Forward'
wide_creators['forward_subtype'] = 'Wide Forward'

forward_ratings = pd.concat([central_forwards, wide_creators], ignore_index=True)

forward_ratings.to_csv('data/forward_ratings.csv', index=False)
print(f"Saved {len(forward_ratings)} rated forwards to data/forward_ratings.csv")

print(forward_ratings[['Player', 'Squad', 'league', 'forward_subtype', 'rating']]
      .sort_values('rating', ascending=False).head(20).to_string(index=False))

Saved 195 rated forwards to data/forward_ratings.csv
           Player           Squad         league forward_subtype  rating
           Mikael             CRB Brazil Serie B Central Forward    67.6
  Tawfik Bentayeb          Troyes        Ligue 2 Central Forward    64.2
     Ivan Prtajin  Kaiserslautern  2. Bundesliga Central Forward    63.3
     Bruno Santos        Londrina Brazil Serie B Central Forward    63.2
  Oliver McBurnie       Hull City   Championship Central Forward    63.0
  Andrea Adorante         Venezia        Serie B Central Forward    62.8
       Barış Atik       Magdeburg  2. Bundesliga    Wide Forward    62.2
     Isac Lidberg    Darmstadt 98  2. Bundesliga Central Forward    62.0
  Joel Pohjanpalo         Palermo        Serie B Central Forward    61.9
     Ross Stewart     Southampton   Championship Central Forward    61.6
 Mateusz Żukowski       Magdeburg  2. Bundesliga Central Forward    61.2
    Fabio Maistro     Juve Stabia        Serie B    Wide Forward    61.

## Midfielder analysis

Like forwards, midfielders can be very individual on the roles that they play within the team. Especially in the way they are involved in the forward position. Some will play more forward, and others drop back into defence when needed, thereore its important to take this into consideration when giving ratings.

In [174]:
midfielders = df[(df['position_group'] == 'Midfielder') & (df['Min'] >= MIN_MINUTES)].copy()
print(len(midfielders))

463


In [175]:
print(midfielders[['Gls_p90', 'Ast_p90', 'TklW', 'Int', 'creative_output_p90']].describe())

          Gls_p90     Ast_p90        TklW         Int  creative_output_p90
count  463.000000  463.000000  463.000000  463.000000           398.000000
mean     0.115313    0.104773   22.084233   17.749460             1.036683
std      0.107769    0.094268   12.627850   10.699449             0.583148
min      0.000000    0.000000    1.000000    0.000000             0.000000
25%      0.040000    0.040000   13.000000    9.000000             0.600000
50%      0.090000    0.080000   20.000000   16.000000             0.900000
75%      0.170000    0.160000   29.000000   24.000000             1.400000
max      0.600000    0.480000   71.000000   72.000000             3.500000


### Splitting midfielders into sub-roles

The spreadin tackles (1 to 71) and creative output (0 to 3.5) within a single group of midfielders shows high levels of diversity. K-means clustering will find natural groups inside this MF group, and help us with understanding the broad group a little better.

In [176]:
midfield_cluster_features = ['TklW', 'Int', 'Gls_p90', 'Ast_p90', 'creative_output_p90']

midfield_cluster_data = midfielders[midfield_cluster_features].fillna(0)

scaler_mf = StandardScaler()
midfield_cluster_scaled = scaler_mf.fit_transform(midfield_cluster_data)

kmeans_mf = KMeans(n_clusters=2, random_state=42, n_init=10)
midfielders['cluster'] = kmeans_mf.fit_predict(midfield_cluster_scaled)

print(midfielders['cluster'].value_counts())
print()
print(midfielders.groupby('cluster')[midfield_cluster_features].mean())

cluster
0    294
1    169
Name: count, dtype: int64

              TklW        Int   Gls_p90   Ast_p90  creative_output_p90
cluster                                                               
0        15.275510  11.588435  0.142993  0.120102             1.122594
1        33.928994  28.467456  0.067160  0.078107             0.907547


Cluster 1 here being the defensive type, with more than double the overall tackles- proving high defensive involvement. Compared to the cluster 0 with muhc higher goal involvement per 90 minutes, and an overall more creative output.

### Different roles within the position

In [177]:
midfielders['TklW_p90'] = midfielders['TklW'] / midfielders['90s']
midfielders['Int_p90'] = midfielders['Int'] / midfielders['90s']
midfielders['Fls_p90'] = midfielders['Fls'] / midfielders['90s']
midfielders['CrdR_p90'] = midfielders['CrdR'] / midfielders['90s']
midfielders['BigChancesCreated_p90'] = midfielders['BigChancesCreated'] / midfielders['90s']

zero_fill_cols_mf = ['BigChancesCreated_p90', 'creative_output_p90']
for col in zero_fill_cols_mf:
    midfielders[col] = midfielders[col].fillna(0)

print(midfielders[['TklW_p90', 'Int_p90', 'Fls_p90', 'CrdR_p90', 'BigChancesCreated_p90', 'creative_output_p90']].describe())

         TklW_p90     Int_p90     Fls_p90    CrdR_p90  BigChancesCreated_p90  \
count  463.000000  463.000000  463.000000  463.000000             463.000000   
mean     1.061970    0.859892    1.324154    0.006798               0.162483   
std      0.431393    0.402351    0.535715    0.020617               0.164644   
min      0.088496    0.000000    0.162602    0.000000               0.000000   
25%      0.777994    0.550862    0.944139    0.000000               0.000000   
50%      1.025641    0.836820    1.263158    0.000000               0.127660   
75%      1.296018    1.116213    1.649985    0.000000               0.239323   
max      2.477876    2.276423    3.645833    0.178571               1.021898   

       creative_output_p90  
count           463.000000  
mean              0.891145  
std               0.649765  
min               0.000000  
25%               0.500000  
50%               0.800000  
75%               1.250000  
max               3.500000  


In [178]:
attacking_mf_criteria = {
    'Gls_p90': 'profit',
    'Ast_p90': 'profit',
    'creative_output_p90': 'profit',
    'BigChancesCreated_p90': 'profit',
    'pct_of_team_goals': 'profit',
    'pct_of_team_assists': 'profit',
    'PassSuccess_pct': 'profit',
    'passing_volume_p90': 'profit',
    'league_success_score': 'profit',
    'CrdR_p90': 'cost',
}

defensive_mf_criteria = {
    'TklW_p90': 'profit',
    'Int_p90': 'profit',
    'pct_of_team_tackles': 'profit',
    'pct_of_team_interceptions': 'profit',
    'tackles_per_100_opp_shots': 'profit',
    'PassSuccess_pct': 'profit',
    'passing_volume_p90': 'profit',
    'long_balls_p90': 'profit',
    'league_success_score': 'profit',
    'CrdR_p90': 'cost',
    'Fls_p90': 'cost',
}

print(f"Attacking Midfielder: {len(attacking_mf_criteria)} criteria")
print(f"Defensive Midfielder: {len(defensive_mf_criteria)} criteria")

Attacking Midfielder: 10 criteria
Defensive Midfielder: 11 criteria


In [179]:
attacking_mfs = midfielders[midfielders['cluster'] == 0].copy()
defensive_mfs = midfielders[midfielders['cluster'] == 1].copy()

In [180]:
for col in ['PassSuccess_pct', 'passing_volume_p90', 'long_balls_p90']:
    attacking_mfs[col] = attacking_mfs[col].fillna(attacking_mfs[col].median())
    defensive_mfs[col] = defensive_mfs[col].fillna(defensive_mfs[col].median())

print(attacking_mfs[['PassSuccess_pct', 'passing_volume_p90', 'long_balls_p90']].isna().sum())
print(defensive_mfs[['PassSuccess_pct', 'passing_volume_p90', 'long_balls_p90']].isna().sum())


PassSuccess_pct       0
passing_volume_p90    0
long_balls_p90        0
dtype: int64
PassSuccess_pct       0
passing_volume_p90    0
long_balls_p90        0
dtype: int64


In [181]:
for col, criterion_type in attacking_mf_criteria.items():
    attacking_mfs[col + '_norm'] = normalize_column(attacking_mfs[col], criterion_type)

for col, criterion_type in defensive_mf_criteria.items():
    defensive_mfs[col + '_norm'] = normalize_column(defensive_mfs[col], criterion_type)

print(attacking_mfs[[c + '_norm' for c in attacking_mf_criteria.keys()]].describe())

       Gls_p90_norm  Ast_p90_norm  creative_output_p90_norm  \
count    294.000000    294.000000                294.000000   
mean       0.238322      0.250213                  0.260739   
std        0.196417      0.215653                  0.207704   
min        0.000000      0.000000                  0.000000   
25%        0.104167      0.104167                  0.114286   
50%        0.200000      0.187500                  0.228571   
75%        0.345833      0.375000                  0.371429   
max        1.000000      1.000000                  1.000000   

       BigChancesCreated_p90_norm  pct_of_team_goals_norm  \
count                  294.000000              294.000000   
mean                     0.180064                0.157655   
std                      0.179043                0.155954   
min                      0.000000                0.000000   
25%                      0.000000                0.054348   
50%                      0.158800                0.116848   
75%  

In [182]:
print(defensive_mfs[[c + '_norm' for c in defensive_mf_criteria.keys()]].describe())

       TklW_p90_norm  Int_p90_norm  pct_of_team_tackles_norm  \
count     169.000000    169.000000                169.000000   
mean        0.455489      0.417204                  0.382013   
std         0.204721      0.171747                  0.179762   
min         0.000000      0.000000                  0.000000   
25%         0.314954      0.305471                  0.246988   
50%         0.416705      0.390273                  0.367470   
75%         0.562203      0.519899                  0.493976   
max         1.000000      1.000000                  1.000000   

       pct_of_team_interceptions_norm  tackles_per_100_opp_shots_norm  \
count                      169.000000                      169.000000   
mean                         0.353832                        0.301630   
std                          0.174722                        0.165690   
min                          0.000000                        0.000000   
25%                          0.232143                     

In [183]:
attacking_mfs['rating'] = (calculate_topsis(attacking_mfs, attacking_mf_criteria) * 100).round(1)
defensive_mfs['rating'] = (calculate_topsis(defensive_mfs, defensive_mf_criteria) * 100).round(1)

print("Top 15 Attacking Midfielders:")
print(attacking_mfs[['Player', 'Squad', 'league', 'Gls_p90', 'Ast_p90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

print("\nTop 15 Defensive Midfielders:")
print(defensive_mfs[['Player', 'Squad', 'league', 'TklW_p90', 'Int_p90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

Top 15 Attacking Midfielders:
            Player            Squad         league  Gls_p90  Ast_p90  rating
      Giacomo Calò        Frosinone        Serie B     0.28     0.43    66.7
            Romulo    Novorizontino Brazil Serie B     0.29     0.44    65.3
   Marvin Wanitzek       Karlsruher  2. Bundesliga     0.44     0.21    61.6
       Rafael Gava    Botafogo (SP) Brazil Serie B     0.18     0.45    59.2
   Antonio Palumbo          Palermo        Serie B     0.11     0.37    58.8
Marquinhos Gabriel        Vila Nova Brazil Serie B     0.16     0.32    58.1
        Kike Pérez          Venezia        Serie B     0.10     0.23    57.6
       Teddy Teuma            Reims        Ligue 2     0.35     0.44    57.4
  Tom Zimmerschied       Elversberg  2. Bundesliga     0.13     0.33    56.1
     Téji Savanier      Montpellier        Ligue 2     0.32     0.19    55.5
   Simone Pontisso        Catanzaro        Serie B     0.13     0.20    55.1
    Adil Aouchiche       Schalke 04  2. Bundes

### Saving results

In [184]:
attacking_mfs['midfielder_subtype'] = 'Attacking Midfielder'
defensive_mfs['midfielder_subtype'] = 'Defensive Midfielder'

midfielder_ratings = pd.concat([attacking_mfs, defensive_mfs], ignore_index=True)

midfielder_ratings.to_csv('data/midfielder_ratings.csv', index=False)
print(f"Saved {len(midfielder_ratings)} rated midfielders to data/midfielder_ratings.csv")

print(midfielder_ratings[['Player', 'Squad', 'league', 'midfielder_subtype', 'rating']]
      .sort_values('rating', ascending=False).head(20).to_string(index=False))

Saved 463 rated midfielders to data/midfielder_ratings.csv
            Player         Squad         league   midfielder_subtype  rating
      Enzo Leopold   Hannover 96  2. Bundesliga Defensive Midfielder    71.1
      Aidan Morris Middlesbrough   Championship Defensive Midfielder    66.7
      Giacomo Calò     Frosinone        Serie B Attacking Midfielder    66.7
     Azor Matusiwa  Ipswich Town   Championship Defensive Midfielder    65.9
            Romulo Novorizontino Brazil Serie B Attacking Midfielder    65.3
       Matt Grimes Coventry City   Championship Defensive Midfielder    64.4
        Theo Leoni         Reims        Ligue 2 Defensive Midfielder    64.0
    Gianluca Busio       Venezia        Serie B Defensive Midfielder    63.0
   Marvin Wanitzek    Karlsruher  2. Bundesliga Attacking Midfielder    61.6
     Caspar Jander   Southampton   Championship Defensive Midfielder    61.6
  Alexandre Lauray       Le Mans        Ligue 2 Defensive Midfielder    61.2
   Dylan Louiserr

In [185]:
progressive_mfs = attacking_mfs
holding_mfs = defensive_mfs

progressive_mfs['midfielder_subtype'] = 'Progressive Midfielder'
holding_mfs['midfielder_subtype'] = 'Holding Midfielder'

midfielder_ratings = pd.concat([progressive_mfs, holding_mfs], ignore_index=True)
midfielder_ratings.to_csv('data/midfielder_ratings.csv', index=False)

print(f"Saved {len(midfielder_ratings)} rated midfielders to data/midfielder_ratings.csv")
print(midfielder_ratings[['Player', 'Squad', 'league', 'midfielder_subtype', 'rating']]
      .sort_values('rating', ascending=False).head(20).to_string(index=False))

Saved 463 rated midfielders to data/midfielder_ratings.csv
            Player         Squad         league     midfielder_subtype  rating
      Enzo Leopold   Hannover 96  2. Bundesliga     Holding Midfielder    71.1
      Aidan Morris Middlesbrough   Championship     Holding Midfielder    66.7
      Giacomo Calò     Frosinone        Serie B Progressive Midfielder    66.7
     Azor Matusiwa  Ipswich Town   Championship     Holding Midfielder    65.9
            Romulo Novorizontino Brazil Serie B Progressive Midfielder    65.3
       Matt Grimes Coventry City   Championship     Holding Midfielder    64.4
        Theo Leoni         Reims        Ligue 2     Holding Midfielder    64.0
    Gianluca Busio       Venezia        Serie B     Holding Midfielder    63.0
   Marvin Wanitzek    Karlsruher  2. Bundesliga Progressive Midfielder    61.6
     Caspar Jander   Southampton   Championship     Holding Midfielder    61.6
  Alexandre Lauray       Le Mans        Ligue 2     Holding Midfielder  

## Attacking and defensive Midfielder rating

In [186]:
am_final = df[(df['position_group'] == 'Attacking Midfielder') & (df['Min'] >= MIN_MINUTES)].copy()
dm_final = df[(df['position_group'] == 'Defensive Midfielder') & (df['Min'] >= MIN_MINUTES)].copy()

print(f"Attacking Midfielder (MF,FW tag): {len(am_final)} players")
print(f"Defensive Midfielder (MF,DF/DF,MF tag): {len(dm_final)} players")

Attacking Midfielder (MF,FW tag): 110 players
Defensive Midfielder (MF,DF/DF,MF tag): 75 players


In [187]:
am_final['TklW_p90'] = am_final['TklW'] / am_final['90s']
am_final['CrdR_p90'] = am_final['CrdR'] / am_final['90s']
am_final['BigChancesCreated_p90'] = am_final['BigChancesCreated'] / am_final['90s']
am_final['Crs_p90'] = am_final['Crs'] / am_final['90s']
am_final['Fld_p90'] = am_final['Fld'] / am_final['90s']

dm_final['TklW_p90'] = dm_final['TklW'] / dm_final['90s']
dm_final['Int_p90'] = dm_final['Int'] / dm_final['90s']
dm_final['Fls_p90'] = dm_final['Fls'] / dm_final['90s']
dm_final['CrdR_p90'] = dm_final['CrdR'] / dm_final['90s']

print("Done")

Done


### Building criteria 

In [188]:
am_final_criteria = {
    'Gls_p90': 'profit',
    'Ast_p90': 'profit',
    'creative_output_p90': 'profit',
    'BigChancesCreated_p90': 'profit',
    'passing_volume_p90': 'profit',
    'PassSuccess_pct': 'profit',
    'pct_of_team_goals': 'profit',
    'pct_of_team_assists': 'profit',
    'Crs_p90': 'profit',
    'Fld_p90': 'profit',
    'league_success_score': 'profit',
    'CrdR_p90': 'cost',
}

dm_final_criteria = {
    'TklW_p90': 'profit',
    'Int_p90': 'profit',
    'pct_of_team_tackles': 'profit',
    'pct_of_team_interceptions': 'profit',
    'tackles_per_100_opp_shots': 'profit',
    'PassSuccess_pct': 'profit',
    'passing_volume_p90': 'profit',
    'long_balls_p90': 'profit',
    'Gls_p90': 'profit',
    'Ast_p90': 'profit',
    'league_success_score': 'profit',
    'CrdR_p90': 'cost',
    'Fls_p90': 'cost',
}

print(f"Attacking Midfielder (tag): {len(am_final_criteria)} criteria")
print(f"Defensive Midfielder (tag): {len(dm_final_criteria)} criteria")


Attacking Midfielder (tag): 12 criteria
Defensive Midfielder (tag): 13 criteria


### Normalising

In [189]:
print(am_final[['PassSuccess_pct', 'passing_volume_p90', 'Crs_p90']].isna().sum())
print(dm_final[['PassSuccess_pct', 'passing_volume_p90', 'long_balls_p90']].isna().sum())

PassSuccess_pct       10
passing_volume_p90    10
Crs_p90                0
dtype: int64
PassSuccess_pct       15
passing_volume_p90    15
long_balls_p90        15
dtype: int64


In [190]:
for col in ['PassSuccess_pct', 'passing_volume_p90']:
    am_final[col] = am_final[col].fillna(am_final[col].median())

for col in ['PassSuccess_pct', 'passing_volume_p90', 'long_balls_p90']:
    dm_final[col] = dm_final[col].fillna(dm_final[col].median())

print(am_final[['PassSuccess_pct', 'passing_volume_p90']].isna().sum())
print(dm_final[['PassSuccess_pct', 'passing_volume_p90', 'long_balls_p90']].isna().sum())

PassSuccess_pct       0
passing_volume_p90    0
dtype: int64
PassSuccess_pct       0
passing_volume_p90    0
long_balls_p90        0
dtype: int64


In [191]:
for col, criterion_type in am_final_criteria.items():
    am_final[col + '_norm'] = normalize_column(am_final[col], criterion_type)

for col, criterion_type in dm_final_criteria.items():
    dm_final[col + '_norm'] = normalize_column(dm_final[col], criterion_type)

print("Attacking Midfielder (tag) normalized stats:")
print(am_final[[c + '_norm' for c in am_final_criteria.keys()]].describe())

print("\nDefensive Midfielder (tag) normalized stats:")
print(dm_final[[c + '_norm' for c in dm_final_criteria.keys()]].describe())

Attacking Midfielder (tag) normalized stats:
       Gls_p90_norm  Ast_p90_norm  creative_output_p90_norm  \
count    110.000000    110.000000                103.000000   
mean       0.319879      0.256740                  0.338592   
std        0.206579      0.175973                  0.207570   
min        0.000000      0.000000                  0.000000   
25%        0.163333      0.120690                  0.208333   
50%        0.286667      0.258621                  0.333333   
75%        0.440000      0.344828                  0.458333   
max        1.000000      1.000000                  1.000000   

       BigChancesCreated_p90_norm  passing_volume_p90_norm  \
count                   96.000000               110.000000   
mean                     0.294150                 0.426595   
std                      0.205470                 0.230017   
min                      0.000000                 0.000000   
25%                      0.166056                 0.259322   
50%            

In [192]:
am_final['rating'] = (calculate_topsis(am_final, am_final_criteria) * 100).round(1)
dm_final['rating'] = (calculate_topsis(dm_final, dm_final_criteria) * 100).round(1)

print("Top 15 Attacking Midfielders (MF,FW):")
print(am_final[['Player', 'Squad', 'league', 'Gls_p90', 'Ast_p90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

print("\nTop 15 Defensive Midfielders (MF,DF):")
print(dm_final[['Player', 'Squad', 'league', 'TklW_p90', 'Int_p90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

Top 15 Attacking Midfielders (MF,FW):
                Player         Squad         league  Gls_p90  Ast_p90  rating
           Léo Scienza   Southampton   Championship     0.26     0.37    64.2
    Chrystian Barletta  Sport Recife Brazil Serie B     0.55     0.21    63.6
      Antonin Bobichon        Pau FC        Ligue 2     0.38     0.25    59.9
          Fabian Reese    Hertha BSC  2. Bundesliga     0.32     0.42    59.8
        Martin Adeline        Troyes        Ligue 2     0.37     0.37    58.6
         Adailson Dadá           CRB Brazil Serie B     0.32     0.24    56.3
       Florent Muslija    Düsseldorf  2. Bundesliga     0.15     0.15    55.9
    Abdoul Kader Bamba Clermont Foot        Ligue 2     0.36     0.13    55.8
        Julian Justvan      Nürnberg  2. Bundesliga     0.23     0.26    54.9
      Augustine Boakye Saint-Étienne        Ligue 2     0.18     0.33    54.8
Alexander Bernhardsson Holstein Kiel  2. Bundesliga     0.13     0.58    54.0
        Nordine Kandil    

In [193]:
am_final['midfielder_subtype'] = 'Attacking Midfielder (MF,FW)'
dm_final['midfielder_subtype'] = 'Defensive Midfielder (MF,DF)'

tagged_mf_ratings = pd.concat([am_final, dm_final], ignore_index=True)
tagged_mf_ratings.to_csv('data/tagged_midfielder_ratings.csv', index=False)

print(f"Saved {len(tagged_mf_ratings)} rated tagged midfielders to data/tagged_midfielder_ratings.csv")
print(tagged_mf_ratings[['Player', 'Squad', 'league', 'midfielder_subtype', 'rating']]
      .sort_values('rating', ascending=False).head(20).to_string(index=False))

Saved 185 rated tagged midfielders to data/tagged_midfielder_ratings.csv
                Player          Squad         league           midfielder_subtype  rating
           Léo Scienza    Southampton   Championship Attacking Midfielder (MF,FW)    64.2
    Chrystian Barletta   Sport Recife Brazil Serie B Attacking Midfielder (MF,FW)    63.6
      Antonin Bobichon         Pau FC        Ligue 2 Attacking Midfielder (MF,FW)    59.9
          Fabian Reese     Hertha BSC  2. Bundesliga Attacking Midfielder (MF,FW)    59.8
        Martin Adeline         Troyes        Ligue 2 Attacking Midfielder (MF,FW)    58.6
          Mehmet Aydin           BTSV  2. Bundesliga Defensive Midfielder (MF,DF)    58.1
          Kai Klefisch   Darmstadt 98  2. Bundesliga Defensive Midfielder (MF,DF)    57.7
       Ethan Galbraith   Swansea City   Championship Defensive Midfielder (MF,DF)    56.3
         Adailson Dadá            CRB Brazil Serie B Attacking Midfielder (MF,FW)    56.3
       Florent Muslija     

## Defender Analysis

In [194]:
defenders = df[(df['position_group'] == 'Defender') & (df['Min'] >= MIN_MINUTES)].copy()
print(len(defenders))

print(defenders[['TklW', 'Int', 'Crs', 'Ast_p90', 'creative_output_p90']].describe())

502
             TklW         Int         Crs     Ast_p90  creative_output_p90
count  502.000000  502.000000  502.000000  502.000000           425.000000
mean    22.005976   24.762948   29.167331    0.049641             0.471059
std     12.331983   13.612897   43.301753    0.067725             0.375908
min      0.000000    0.000000    0.000000    0.000000             0.000000
25%     13.000000   15.000000    3.000000    0.000000             0.200000
50%     20.000000   22.000000   12.000000    0.030000             0.400000
75%     28.000000   32.000000   37.000000    0.077500             0.600000
max     72.000000   80.000000  273.000000    0.390000             2.400000


### Kmeans clustering for sub-roles within the DF category

In [195]:
defender_cluster_features = ['TklW', 'Int', 'Crs', 'Ast_p90', 'creative_output_p90']

defender_cluster_data = defenders[defender_cluster_features].fillna(0)

scaler_df = StandardScaler()
defender_cluster_scaled = scaler_df.fit_transform(defender_cluster_data)

kmeans_df = KMeans(n_clusters=2, random_state=42, n_init=10)
defenders['cluster'] = kmeans_df.fit_predict(defender_cluster_scaled)

print(defenders['cluster'].value_counts())
print()
print(defenders.groupby('cluster')[defender_cluster_features].mean())

cluster
1    426
0     76
Name: count, dtype: int64

              TklW        Int         Crs   Ast_p90  creative_output_p90
cluster                                                                 
0        33.881579  29.565789  106.500000  0.136711             1.098667
1        19.887324  23.906103   15.370892  0.034108             0.336571


Cluster 0 (76 players) crosses far more (106.5 vs 15.4) and creates more (1.10 vs 0.34) — genuine Wingbacks. Cluster 1 (426 players) is the larger, more defensively-focused Centre-Back group. The split is real, matching the original concern that one broad "Defender" tag was hiding two different jobs.

Before carrying on though, there are a few things we need to sort address:

1. TklW_p90, Int_p90, Crs_p90 don't exist yet for defenders
2. OG (own goals) and team_clean_sheets is a raw total, whihc needs to be /90 for the games played bias



### Adressing Issues

In [196]:
defenders['TklW_p90'] = defenders['TklW'] / defenders['90s']
defenders['Int_p90'] = defenders['Int'] / defenders['90s']
defenders['Crs_p90'] = defenders['Crs'] / defenders['90s']
defenders['OG_p90'] = defenders['OG'] / defenders['90s']
defenders['CrdR_p90'] = defenders['CrdR'] / defenders['90s']


league_match_counts = {
    'Championship': 46,
    '2. Bundesliga': 34,
    'Ligue 2': 34,
    'Serie B': 38,
    'Brazil Serie B': 38,
}
defenders['league_matches'] = defenders['league'].map(league_match_counts)
defenders['clean_sheet_rate'] = defenders['team_clean_sheets'] / defenders['league_matches']

print(defenders[['TklW_p90', 'Int_p90', 'Crs_p90', 'OG_p90', 'clean_sheet_rate', 'CrdR_p90']].describe())

         TklW_p90     Int_p90     Crs_p90      OG_p90  clean_sheet_rate  \
count  502.000000  502.000000  502.000000  502.000000        502.000000   
mean     1.015834    1.128539    1.348302    0.006554          0.252103   
std      0.416794    0.392395    1.772364    0.021379          0.092158   
min      0.000000    0.000000    0.000000    0.000000          0.052632   
25%      0.720000    0.881508    0.133351    0.000000          0.184211   
50%      0.976527    1.095415    0.527309    0.000000          0.239130   
75%      1.248948    1.349872    1.895425    0.000000          0.304348   
max      2.545455    2.602740    8.974359    0.240000          0.473684   

         CrdR_p90  
count  502.000000  
mean     0.012417  
std      0.026846  
min      0.000000  
25%      0.000000  
50%      0.000000  
75%      0.000000  
max      0.198020  


### DF group dictionary

In [197]:
wingback_criteria = {
    'Crs_p90': 'profit',
    'Ast_p90': 'profit',
    'creative_output_p90': 'profit',
    'PassSuccess_pct': 'profit',
    'pct_of_team_assists': 'profit',
    'TklW_p90': 'profit',
    'Int_p90': 'profit',
    'league_success_score': 'profit',
    'CrdR_p90': 'cost',
    'OG_p90': 'cost',
}

centreback_criteria = {
    'TklW_p90': 'profit',
    'Int_p90': 'profit',
    'pct_of_team_tackles': 'profit',
    'pct_of_team_interceptions': 'profit',
    'tackles_per_100_opp_shots': 'profit',
    'PassSuccess_pct': 'profit',
    'passing_volume_p90': 'profit',
    'long_balls_p90': 'profit',
    'clean_sheet_rate': 'profit',
    'league_success_score': 'profit',
    'CrdR_p90': 'cost',
    'OG_p90': 'cost',
}

print(f"Wingback: {len(wingback_criteria)} criteria")
print(f"Centre-Back: {len(centreback_criteria)} criteria")

Wingback: 10 criteria
Centre-Back: 12 criteria


### Checks for missing data ahead of normalisation

In [198]:
wingbacks = defenders[defenders['cluster'] == 0].copy()
centrebacks = defenders[defenders['cluster'] == 1].copy()

print(f"Wingbacks: {len(wingbacks)}")
print(f"Centre-Backs: {len(centrebacks)}")

print(wingbacks[list(wingback_criteria.keys())].isna().sum())
print()
print(centrebacks[list(centreback_criteria.keys())].isna().sum())

Wingbacks: 76
Centre-Backs: 426
Crs_p90                 0
Ast_p90                 0
creative_output_p90     1
PassSuccess_pct         1
pct_of_team_assists     0
TklW_p90                0
Int_p90                 0
league_success_score    0
CrdR_p90                0
OG_p90                  0
dtype: int64

TklW_p90                       0
Int_p90                        0
pct_of_team_tackles            0
pct_of_team_interceptions      0
tackles_per_100_opp_shots      0
PassSuccess_pct              105
passing_volume_p90           105
long_balls_p90               105
clean_sheet_rate               0
league_success_score           0
CrdR_p90                       0
OG_p90                         0
dtype: int64


In [199]:
for col in ['creative_output_p90', 'PassSuccess_pct']:
    wingbacks[col] = wingbacks[col].fillna(wingbacks[col].median())

for col in ['PassSuccess_pct', 'passing_volume_p90', 'long_balls_p90']:
    centrebacks[col] = centrebacks[col].fillna(centrebacks[col].median())

print(wingbacks[list(wingback_criteria.keys())].isna().sum())
print(centrebacks[list(centreback_criteria.keys())].isna().sum())

Crs_p90                 0
Ast_p90                 0
creative_output_p90     0
PassSuccess_pct         0
pct_of_team_assists     0
TklW_p90                0
Int_p90                 0
league_success_score    0
CrdR_p90                0
OG_p90                  0
dtype: int64
TklW_p90                     0
Int_p90                      0
pct_of_team_tackles          0
pct_of_team_interceptions    0
tackles_per_100_opp_shots    0
PassSuccess_pct              0
passing_volume_p90           0
long_balls_p90               0
clean_sheet_rate             0
league_success_score         0
CrdR_p90                     0
OG_p90                       0
dtype: int64


### Normalisation

In [200]:
for col, criterion_type in wingback_criteria.items():
    wingbacks[col + '_norm'] = normalize_column(wingbacks[col], criterion_type)

for col, criterion_type in centreback_criteria.items():
    centrebacks[col + '_norm'] = normalize_column(centrebacks[col], criterion_type)

print("Wingback normalized stats:")
print(wingbacks[[c + '_norm' for c in wingback_criteria.keys()]].describe())

print("\nCentre-Back normalized stats:")
print(centrebacks[[c + '_norm' for c in centreback_criteria.keys()]].describe())

Wingback normalized stats:
       Crs_p90_norm  Ast_p90_norm  creative_output_p90_norm  \
count     76.000000     76.000000                 76.000000   
mean       0.428896      0.350540                  0.314404   
std        0.243170      0.222211                  0.197673   
min        0.000000      0.000000                  0.000000   
25%        0.244774      0.179487                  0.157895   
50%        0.378113      0.320513                  0.263158   
75%        0.598021      0.487179                  0.368421   
max        1.000000      1.000000                  1.000000   

       PassSuccess_pct_norm  pct_of_team_assists_norm  TklW_p90_norm  \
count             76.000000                 76.000000      76.000000   
mean               0.513843                  0.357866       0.357438   
std                0.233247                  0.224874       0.208286   
min                0.000000                  0.000000       0.000000   
25%                0.359375                  

### Topisis calculations

In [201]:
wingbacks['rating'] = (calculate_topsis(wingbacks, wingback_criteria) * 100).round(1)
centrebacks['rating'] = (calculate_topsis(centrebacks, centreback_criteria) * 100).round(1)

print("Top 15 Wingbacks:")
print(wingbacks[['Player', 'Squad', 'league', 'Crs_p90', 'Ast_p90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

print("\nTop 15 Centre-Backs:")
print(centrebacks[['Player', 'Squad', 'league', 'TklW_p90', 'Int_p90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

Top 15 Wingbacks:
            Player           Squad        league  Crs_p90  Ast_p90  rating
 Maximilian Wittek          Bochum 2. Bundesliga 7.095588     0.26    66.7
    Tim Handwerker         Arminia 2. Bundesliga 8.923077     0.36    66.0
Evans Jean-Lambert        Rodez AF       Ligue 2 7.992832     0.32    64.4
     Alfie Doughty        Millwall  Championship 8.974359     0.32    61.9
        Josh Tymon    Swansea City  Championship 5.464853     0.20    61.1
      Nolan Galves        Rodez AF       Ligue 2 7.345455     0.25    60.0
     Berkay Yılmaz        Nürnberg 2. Bundesliga 4.223602     0.22    59.3
Alexander Rossipal         Dresden 2. Bundesliga 6.513410     0.27    59.0
   Milan van Ewijk   Coventry City  Championship 2.595238     0.19    58.7
     Lasse Günther      Elversberg 2. Bundesliga 3.631285     0.39    58.2
          Joe Ward    Derby County  Championship 7.055215     0.25    58.1
      Ryan Manning     Southampton  Championship 7.647059     0.17    57.4
       

In [202]:
wingbacks['defender_subtype'] = 'Wingback'
centrebacks['defender_subtype'] = 'Centre-Back'

defender_ratings = pd.concat([wingbacks, centrebacks], ignore_index=True)
defender_ratings.to_csv('data/defender_ratings.csv', index=False)

print(f"Saved {len(defender_ratings)} rated defenders to data/defender_ratings.csv")
print(defender_ratings[['Player', 'Squad', 'league', 'defender_subtype', 'rating']]
      .sort_values('rating', ascending=False).head(20).to_string(index=False))

Saved 502 rated defenders to data/defender_ratings.csv
            Player         Squad         league defender_subtype  rating
    Adrien Monfray        Troyes        Ligue 2      Centre-Back    69.8
 Maximilian Wittek        Bochum  2. Bundesliga         Wingback    66.7
      Harold Voyer       Le Mans        Ligue 2      Centre-Back    66.4
   Maximilian Rohr    Elversberg  2. Bundesliga      Centre-Back    66.3
    Tim Handwerker       Arminia  2. Bundesliga         Wingback    66.0
   Michael Svoboda       Venezia        Serie B      Centre-Back    65.9
       Rodrigo Sam     Juventude Brazil Serie B      Centre-Back    65.4
Evans Jean-Lambert      Rodez AF        Ligue 2         Wingback    64.4
     Giovanni Zaro        Cesena        Serie B      Centre-Back    63.8
   Marcus Mathisen     Magdeburg  2. Bundesliga      Centre-Back    63.7
      Mickael Nade Saint-Étienne        Ligue 2      Centre-Back    63.6
       Mattia Bani       Palermo        Serie B      Centre-Back    6

## Goalkeeper Analysis

Goalkeeper is a well-defined role, as a result there is no need for clustering the data into groups. This uses data from the all_goalkeepers.csv to form an accurate rating for each player in this position. 

In [203]:
goalkeepers = pd.read_csv('data/all_goalkeepers.csv')

MIN_MATCHES = 10
qualified_gks = goalkeepers[goalkeepers['MP'] >= MIN_MATCHES].copy()

print(len(qualified_gks))
print(qualified_gks[['Save_pct', 'CS%', 'GA90', 'PK_Save_pct']].describe())

121
         Save_pct         CS%        GA90  PK_Save_pct
count  121.000000  121.000000  121.000000   109.000000
mean    67.945455   26.813223    1.320744    18.055046
std      6.118115   11.423659    0.343614    25.015749
min     48.900000    0.000000    0.470000     0.000000
25%     64.300000   18.400000    1.080000     0.000000
50%     67.700000   25.000000    1.320000     0.000000
75%     71.900000   33.300000    1.500000    33.300000
max     86.000000   70.600000    2.800000   100.000000


This nect section should help improve the goalkeeper dataset with team-level context by merging each goalkeeper with their club's league_success_score. It also calculates saves per 90 minutes (Saves_p90) to standardise shot-stopping performance across different playing times. Finally, summary statistics are displayed for both Saves_p90 and league_success_score to provide an overview of their distributions before further analysis.

In [204]:
gk_context = df[['Squad', 'league', 'league_success_score']].drop_duplicates()
qualified_gks = qualified_gks.merge(gk_context, on=['Squad', 'league'], how='left')

qualified_gks['Saves_p90'] = qualified_gks['Saves'] / qualified_gks['90s']

print(qualified_gks[['Saves_p90', 'league_success_score']].describe())

        Saves_p90  league_success_score
count  121.000000            121.000000
mean     2.790989              0.472653
std      0.542675              0.308775
min      1.655172              0.000000
25%      2.446809              0.211000
50%      2.810127              0.471000
75%      3.142857              0.737000
max      4.315789              1.000000


### GK Dictionary

In [205]:
goalkeeper_criteria = {
    'Save_pct': 'profit',
    'CS%': 'profit',
    'PK_Save_pct': 'profit',
    'Saves_p90': 'profit',
    'league_success_score': 'profit',
    'GA90': 'cost',
}

print(f"Goalkeeper: {len(goalkeeper_criteria)} criteria")

Goalkeeper: 6 criteria


In [206]:
qualified_gks['PK_Save_pct'] = qualified_gks['PK_Save_pct'].fillna(qualified_gks['PK_Save_pct'].median())

print(qualified_gks[list(goalkeeper_criteria.keys())].isna().sum())

Save_pct                0
CS%                     0
PK_Save_pct             0
Saves_p90               0
league_success_score    0
GA90                    0
dtype: int64


### Data Normalisation

In [207]:
for col, criterion_type in goalkeeper_criteria.items():
    qualified_gks[col + '_norm'] = normalize_column(qualified_gks[col], criterion_type)

print(qualified_gks[[c + '_norm' for c in goalkeeper_criteria.keys()]].describe())

       Save_pct_norm    CS%_norm  PK_Save_pct_norm  Saves_p90_norm  \
count     121.000000  121.000000        121.000000      121.000000   
mean        0.513355    0.379791          0.162645        0.426900   
std         0.164909    0.161808          0.243429        0.203966   
min         0.000000    0.000000          0.000000        0.000000   
25%         0.415094    0.260623          0.000000        0.297539   
50%         0.506739    0.354108          0.000000        0.434093   
75%         0.619946    0.471671          0.250000        0.559150   
max         1.000000    1.000000          1.000000        1.000000   

       league_success_score_norm   GA90_norm  
count                 121.000000  121.000000  
mean                    0.472653    0.634874  
std                     0.308775    0.147474  
min                     0.000000    0.000000  
25%                     0.211000    0.557940  
50%                     0.471000    0.635193  
75%                     0.737000    0.73

### TOPISIS Calculations

In [208]:
qualified_gks['rating'] = (calculate_topsis(qualified_gks, goalkeeper_criteria) * 100).round(1)

print("Top 15 Goalkeepers:")
print(qualified_gks[['Player', 'Squad', 'league', 'Save_pct', 'CS%', 'GA90', 'rating']]
      .sort_values('rating', ascending=False).head(15).to_string(index=False))

Top 15 Goalkeepers:
              Player        Squad         league  Save_pct  CS%  GA90  rating
   Lorenzo Palmisani    Frosinone        Serie B      80.1 39.5  0.90    75.3
             Jandrei    Juventude Brazil Serie B      86.0 70.6  0.47    64.1
       Jesse Joronen      Palermo        Serie B      76.9 45.7  0.90    62.8
       Airton Moraes     Criciúma Brazil Serie B      81.6 41.7  0.75    61.1
    Christian Walton Ipswich Town   Championship      72.0 44.4  1.01    59.2
        Loris Karius   Schalke 04  2. Bundesliga      71.4 43.3  0.80    59.0
     Vagner da Silva     Operário Brazil Serie B      75.8 50.0  1.07    58.8
  Leandro Chichizola       Modena        Serie B      73.8 42.4  0.87    57.9
       Quentin Braat     Rodez AF        Ligue 2      75.9 20.6  1.15    57.8
         Demba Thiam        Monza        Serie B      75.4 42.1  0.84    57.8
       Hillel Konaté       Troyes        Ligue 2      67.7 36.4  0.93    57.6
        Max Crocombe     Millwall   Champion

In [209]:
qualified_gks.to_csv('data/goalkeeper_ratings.csv', index=False)
print(f"Saved {len(qualified_gks)} rated goalkeepers to data/goalkeeper_ratings.csv")

Saved 121 rated goalkeepers to data/goalkeeper_ratings.csv


## Sensitivity Analysis

A sensitivity analysis outputs exactly what would make a player better- a real use for football scouting and player training. For a chosen player, the analysis will simulate improving juts one statistic, and outputing their new rating.

In [210]:
def simulate_stat_improvement(df, criteria, player_idx, stat_to_change, pct_increase):
    df_sim = df.copy()
    
    original_value = df_sim.loc[player_idx, stat_to_change]
    criterion_type = criteria[stat_to_change]
    
    # for a cost criterion improvement means decreasing the value
    if criterion_type == 'profit':
        new_value = original_value * (1 + pct_increase)
    else:
        new_value = original_value * (1 - pct_increase)
    
    df_sim.loc[player_idx, stat_to_change] = new_value
    
    for col, ctype in criteria.items():
        df_sim[col + '_norm'] = normalize_column(df_sim[col], ctype)
    
    df_sim['rating_sim'] = (calculate_topsis(df_sim, criteria) * 100).round(2)
    
    return df_sim

print("Function defined")

Function defined


In [211]:
mid_player = central_forwards.sort_values('rating', ascending=False).iloc[75]
print(f"{mid_player['Player']} - rating: {mid_player['rating']}")

mid_idx = central_forwards[central_forwards['Player'] == mid_player['Player']].index[0]

results_mid = []
for stat in central_forward_criteria.keys():
    df_sim = simulate_stat_improvement(central_forwards, central_forward_criteria, mid_idx, stat, 0.10)
    new_rating = df_sim.loc[mid_idx, 'rating_sim']
    change = round(new_rating - mid_player['rating'], 2)
    results_mid.append({'stat': stat, 'new_rating': new_rating, 'rating_change': change})

results_mid_df = pd.DataFrame(results_mid).sort_values('rating_change', ascending=False)
print(results_mid_df.to_string(index=False))

Omar Sadik - rating: 49.7
                stat  new_rating  rating_change
                SoT%       50.27           0.57
               Sh/90       50.10           0.40
             Fld_p90       50.07           0.37
league_success_score       50.06           0.36
             Ast_p90       49.99           0.29
             Gls_p90       49.80           0.10
                G/Sh       49.76           0.06
   pct_of_team_goals       49.76           0.06
            CrdR_p90       49.67          -0.03
     offside_penalty       49.67          -0.03


In [213]:
all_sensitivity_results = []

for idx in central_forwards.index:
    original = central_forwards.loc[idx, 'rating']
    for stat in central_forward_criteria.keys():
        df_sim = simulate_stat_improvement(central_forwards, central_forward_criteria, idx, stat, 0.10)
        new_rating = df_sim.loc[idx, 'rating_sim']
        change = new_rating - original
        all_sensitivity_results.append({'stat': stat, 'rating_change': change})

sensitivity_summary = pd.DataFrame(all_sensitivity_results).groupby('stat')['rating_change'].agg(['mean', 'std']).sort_values('mean', ascending=False)
print(sensitivity_summary)

                          mean       std
stat                                    
SoT%                  0.530980  0.147429
Sh/90                 0.446144  0.137913
league_success_score  0.291373  0.187641
G/Sh                  0.280915  0.134690
Fld_p90               0.278105  0.127861
Gls_p90               0.275556  0.140403
Ast_p90               0.246405  0.161997
pct_of_team_goals     0.236732  0.153828
offside_penalty       0.051634  0.145794
CrdR_p90              0.037974  0.126664


In [214]:
sensitivity_summary.to_csv('data/central_forward_sensitivity.csv')
print("Saved to data/central_forward_sensitivity.csv")

Saved to data/central_forward_sensitivity.csv


In [217]:
PERCENTAGE_STATS = ['PassSuccess_pct', 'SoT%', 'CS%', 'PK_Save_pct', 'LongBallSuccess_pct', 'Save_pct']

def simulate_stat_improvement_v2(df, criteria, player_idx, stat_to_change, pct_increase, fixed_point_increase=3):
    df_sim = df.copy()
    
    original_value = df_sim.loc[player_idx, stat_to_change]
    criterion_type = criteria[stat_to_change]
    
    if stat_to_change in PERCENTAGE_STATS:
        # use a fixed point increase for percentage-based stats, not a relative one
        if criterion_type == 'profit':
            new_value = original_value + fixed_point_increase
        else:
            new_value = original_value - fixed_point_increase
    else:
        if criterion_type == 'profit':
            new_value = original_value * (1 + pct_increase)
        else:
            new_value = original_value * (1 - pct_increase)
    
    df_sim.loc[player_idx, stat_to_change] = new_value
    
    for col, ctype in criteria.items():
        df_sim[col + '_norm'] = normalize_column(df_sim[col], ctype)
    
    df_sim['rating_sim'] = (calculate_topsis(df_sim, criteria) * 100).round(2)
    
    return df_sim

print("Fixed function defined")

Fixed function defined


In [221]:
all_sensitivity_results_wb = []
for idx in wingbacks.index:
    original = wingbacks.loc[idx, 'rating']
    for stat in wingback_criteria.keys():
        df_sim = simulate_stat_improvement_v2(wingbacks, wingback_criteria, idx, stat, 0.10)
        new_rating = df_sim.loc[idx, 'rating_sim']
        change = new_rating - original
        all_sensitivity_results_wb.append({'stat': stat, 'rating_change': change})
sensitivity_summary_wb = pd.DataFrame(all_sensitivity_results_wb).groupby('stat')['rating_change'].agg(['mean', 'std']).sort_values('mean', ascending=False)
print(sensitivity_summary_wb)

                          mean       std
stat                                    
PassSuccess_pct       1.118026  0.236959
TklW_p90              0.492500  0.158300
Int_p90               0.406053  0.156487
creative_output_p90   0.402763  0.153480
league_success_score  0.331184  0.196706
Crs_p90               0.325789  0.158200
pct_of_team_assists   0.243421  0.145406
Ast_p90               0.238289  0.143930
CrdR_p90              0.059868  0.148697
OG_p90                0.017763  0.090474


In [220]:
print("PassSuccess_pct range:", wingbacks['PassSuccess_pct'].min(), "to", wingbacks['PassSuccess_pct'].max())
print("TklW_p90 range:", wingbacks['TklW_p90'].min(), "to", wingbacks['TklW_p90'].max())

# check one real example directly
example_idx = wingbacks.index[0]
original_pass = wingbacks.loc[example_idx, 'PassSuccess_pct']
new_pass = original_pass * 1.10
print(f"\nExample player: PassSuccess_pct goes from {original_pass:.1f} to {new_pass:.1f} (a {new_pass - original_pass:.1f} point jump)")

PassSuccess_pct range: 70.1 to 89.3
TklW_p90 range: 0.6159420289855072 to 2.4827586206896552

Example player: PassSuccess_pct goes from 86.1 to 94.7 (a 8.6 point jump)


In [222]:
sensitivity_summary_wb.to_csv('data/wingback_sensitivity.csv')
print("Saved wingback sensitivity")

all_sensitivity_results_gk = []

for idx in qualified_gks.index:
    original = qualified_gks.loc[idx, 'rating']
    for stat in goalkeeper_criteria.keys():
        df_sim = simulate_stat_improvement_v2(qualified_gks, goalkeeper_criteria, idx, stat, 0.10)
        new_rating = df_sim.loc[idx, 'rating_sim']
        change = new_rating - original
        all_sensitivity_results_gk.append({'stat': stat, 'rating_change': change})

sensitivity_summary_gk = pd.DataFrame(all_sensitivity_results_gk).groupby('stat')['rating_change'].agg(['mean', 'std']).sort_values('mean', ascending=False)
print(sensitivity_summary_gk)

Saved wingback sensitivity
                          mean       std
stat                                    
Saves_p90             1.503884  0.481630
Save_pct              1.117769  0.211922
GA90                  0.867686  0.348089
CS%                   0.567190  0.117843
league_success_score  0.541983  0.383064
PK_Save_pct           0.347025  0.123306


In [223]:
sensitivity_summary_gk.to_csv('data/goalkeeper_sensitivity.csv')
print("Saved goalkeeper sensitivity")

Saved goalkeeper sensitivity
